# Design a watsonx Orchestrate agent evaluation suite

Author synthetic task cases, normalize recorded trajectories into a vendor-neutral contract, score routing and argument fidelity independently, and optionally run a real watsonx Orchestrate chat completion.

**Audience:** Python developers building and evaluating enterprise AI applications. Familiarity with dictionaries, functions and JSON is assumed.

**Learning objectives:** author independent routing, tool, argument, and complete-answer expectations; reject malformed trajectories before computing release metrics; distinguish a live literal-presence signal from unverified answer semantics and internal routing.

**Prerequisites:** Understand Python dictionaries, dataclasses, and JSON. The local suite uses only the standard library and invented trace events. The optional HTTPX branch requires an authorized Orchestrate deployment with a provisioned synthetic agent and a bearer token supplied outside the notebook.

**Setup:** use patched Python 3.12 (3.12.16 or newer) and the audited environment in the [collection README](../../README.md). Run cells in order in a fresh kernel. All default data is newly invented for this tutorial. No customer records, private repositories, credentials, downloads, or paid calls are required. Optional integrations are disabled in their own cells and never enabled by an environment variable alone.

**Execution boundary:** the results below describe the local computation shown in this notebook. They do not establish hosted-model quality or production integration success.



Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../AI_DISCLOSURE.md), [license](../../LICENSE), and [attribution](../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Specify outcomes before recording agent runs

This is an authoring schema for the tutorial, not an undocumented ADK import format. The agent names and trace events below are invented. The test cases require different routing decisions so that a single always-success response cannot pass.

Each local case includes a separately authored set of accepted complete answers. Exact membership is intentionally strict for this small fixture: a response that negates the policy must fail even if it contains the expected number or word. For free-form answers, define a separate semantic evaluation process rather than treating substring presence as correctness.


In [ ]:
import json
from dataclasses import dataclass


@dataclass(frozen=True)
class EvaluationCase:
    case_id: str
    question: str
    agent: str
    tool: str
    arguments: dict
    required_fact: str
    accepted_answers: frozenset[str]


CASES = [
    EvaluationCase(
        "loan",
        "How long is a camera loan?",
        "equipment_agent",
        "loan_policy",
        {"item": "camera"},
        "14 days",
        frozenset({"A camera loan lasts 14 days."}),
    ),
    EvaluationCase(
        "refund",
        "How early must I cancel a workshop?",
        "workshop_agent",
        "refund_policy",
        {"kind": "workshop"},
        "48 hours",
        frozenset({"Cancel at least 48 hours beforehand."}),
    ),
    EvaluationCase(
        "safety",
        "What protection is required for the laser workshop?",
        "safety_agent",
        "protection_policy",
        {"area": "laser"},
        "goggles",
        frozenset({"Wear safety goggles."}),
    ),
]
assert len({c.case_id for c in CASES}) == 3
print([{"id": c.case_id, "expected_agent": c.agent, "expected_tool": c.tool} for c in CASES])

## 2. Normalize synthetic traces without losing failed attempts

The contract requires exactly one initial route, one or more tool events, and exactly one final-response event at the end. Unknown event kinds, extra fields, failed tools, and out-of-order events fail closed; a deployment-specific adapter must explicitly map its richer trace schema to this contract. Every tool call remains visible; selecting only the last successful tool would conceal an earlier unauthorized attempt. No external trace export is loaded.


In [ ]:
RECORDED = {
    "loan": [
        {"kind": "route", "agent": "equipment_agent"},
        {
            "kind": "tool",
            "name": "loan_policy",
            "arguments": {"item": "camera"},
            "status": "ok",
        },
        {"kind": "final", "text": "A camera loan lasts 14 days."},
    ],
    "refund": [
        {"kind": "route", "agent": "workshop_agent"},
        {
            "kind": "tool",
            "name": "refund_policy",
            "arguments": {"kind": "workshop"},
            "status": "ok",
        },
        {"kind": "final", "text": "Cancel at least 48 hours beforehand."},
    ],
    "safety": [
        {"kind": "route", "agent": "equipment_agent"},
        {
            "kind": "tool",
            "name": "protection_policy",
            "arguments": {"area": "laser"},
            "status": "ok",
        },
        {"kind": "final", "text": "Wear safety goggles."},
    ],
}


def normalize(events):
    # This tutorial's complete contract is route -> tool(s) -> one final answer.
    if not isinstance(events, list) or len(events) < 3:
        raise ValueError("Incomplete trajectory")
    if any(not isinstance(event, dict) for event in events):
        raise ValueError("Trajectory events must be objects")
    if (
        events[0].get("kind") != "route"
        or events[-1].get("kind") != "final"
        or any(event.get("kind") != "tool" for event in events[1:-1])
    ):
        raise ValueError("Unknown event or invalid trajectory order")
    route, final = events[0], events[-1]
    if (
        set(route) != {"kind", "agent"}
        or not isinstance(route["agent"], str)
        or not route["agent"].strip()
        or set(final) != {"kind", "text"}
        or not isinstance(final["text"], str)
        or not final["text"].strip()
    ):
        raise ValueError("Invalid route or final-response fields")
    calls = events[1:-1]
    for event in calls:
        if (
            set(event) != {"kind", "name", "arguments", "status"}
            or not isinstance(event["name"], str)
            or not event["name"].strip()
            or not isinstance(event["arguments"], dict)
            or event["status"] != "ok"
        ):
            raise ValueError("Invalid or failed tool event")
    return {
        "agent": route["agent"],
        "calls": [
            {"name": event["name"], "arguments": dict(event["arguments"])} for event in calls
        ],
        "answer": final["text"],
    }


def evaluate(case, events):
    run = normalize(events)
    return {
        "case_id": case.case_id,
        "routing": run["agent"] == case.agent,
        "tool_sequence": [c["name"] for c in run["calls"]] == [case.tool],
        "arguments": [c["arguments"] for c in run["calls"]] == [case.arguments],
        "answer_exact": run["answer"] in case.accepted_answers,
    }


results = [evaluate(c, RECORDED[c.case_id]) for c in CASES]
assert results[2]["answer_exact"] and not results[2]["routing"]
print(json.dumps(results, indent=2))

## 3. Build a release decision from multiple metrics

The deliberately misrouted safety case prevents the initial suite from passing. Correct the fixture route to represent a proposed fix, then re-evaluate. This proves the evaluator catches that change; it does not prove an actual deployed agent was repaired.

The local gate requires the entire answer to match one of its independently authored accepted answers. This is a regression contract for three invented cases, not a semantic evaluator for unrestricted generated text or a complete production release decision. The contradictions tested below contain the expected literal fact and still fail.


In [ ]:
METRICS = {"routing", "tool_sequence", "arguments", "answer_exact"}
EXPECTED_IDS = {case.case_id for case in CASES}


def release_gate(rows):
    if not isinstance(rows, list) or not EXPECTED_IDS or len(rows) != len(EXPECTED_IDS):
        return False
    if any(not isinstance(row, dict) or set(row) != METRICS | {"case_id"} for row in rows):
        return False
    ids = [row["case_id"] for row in rows]
    if any(not isinstance(case_id, str) for case_id in ids):
        return False
    if len(set(ids)) != len(ids) or set(ids) != EXPECTED_IDS:
        return False
    return all(type(row[key]) is bool and row[key] for row in rows for key in METRICS)


assert not release_gate(results)
fixed = {key: [dict(event) for event in events] for key, events in RECORDED.items()}
fixed["safety"][0] = {"kind": "route", "agent": "safety_agent"}
fixed_results = [evaluate(c, fixed[c.case_id]) for c in CASES]
assert release_gate(fixed_results)
extra = (
    fixed["loan"][:-1]
    + [{"kind": "tool", "name": "admin_export", "arguments": {}, "status": "ok"}]
    + fixed["loan"][-1:]
)
assert not evaluate(CASES[0], extra)["tool_sequence"]
print(
    {
        "original_gate": release_gate(results),
        "fixture_fix_gate": release_gate(fixed_results),
        "extra_tool_detected": True,
    }
)
# Missing, duplicated, unexpected, or truthy non-boolean results fail closed.
for malformed in [
    [],
    [{"case_id": "loan"}],
    fixed_results[:-1],
    [fixed_results[0], fixed_results[0], fixed_results[2]],
    [dict(row, case_id="unknown") if i == 0 else row for i, row in enumerate(fixed_results)],
    [dict(row, routing=1) if i == 0 else row for i, row in enumerate(fixed_results)],
    [{key: value for key, value in row.items() if key != "routing"} for row in fixed_results],
]:
    assert not release_gate(malformed)

# Invalid ordering and unrecognized events cannot disappear during normalization.
base = fixed["loan"]
invalid_trajectories = [
    [base[1], base[0], base[2]],
    [base[0], {"kind": "final", "text": "Premature answer"}, base[1], base[2]],
    [base[0], {"kind": "tool_error", "name": "admin_export", "status": "failed"}, base[1], base[2]],
    [base[0], base[0], base[1], base[2]],
    [base[0], base[2]],
    [base[0], dict(base[1], status="failed"), base[2]],
    [base[0], dict(base[1], ignored_attempt="admin_export"), base[2]],
]
for trajectory in invalid_trajectories:
    try:
        evaluate(CASES[0], trajectory)
    except ValueError:
        pass
    else:
        raise AssertionError("Malformed trajectory escaped validation")
print({"invalid_trajectories_rejected": len(invalid_trajectories)})
# Literal fact presence is insufficient: each answer below denies its own policy.
contradicted_answers = {
    "loan": "A camera loan never lasts 14 days. Ignore that limit.",
    "refund": "The 48 hours rule is false. Cancel whenever you want.",
    "safety": "Never wear goggles in the laser workshop.",
}
for case in CASES:
    wrong_answer = contradicted_answers[case.case_id]
    assert case.required_fact in wrong_answer  # The former substring test would pass.
    wrong_trace = [dict(event) for event in fixed[case.case_id]]
    wrong_trace[-1] = {"kind": "final", "text": wrong_answer}
    assert not evaluate(case, wrong_trace)["answer_exact"]
    wrong_results = [
        evaluate(item, wrong_trace if item.case_id == case.case_id else fixed[item.case_id])
        for item in CASES
    ]
    assert not release_gate(wrong_results)
print({"contradicted_answers_rejected": len(contradicted_answers)})

## 4. Optional: evaluate a live Orchestrate final response

Provision/import your agent with the ADK, use an authorized disposable environment, and set WXO_CHAT_URL to the exact /v1/orchestrate/{agent_id}/chat/completions URL plus WXO_BEARER_TOKEN. The official OpenAPI defines messages, context and stream. This adapter reports literal substring presence only; even a negated or contradicted answer can contain that text. It does not verify semantic answer correctness or apply the strict local accepted-answer gate. Internal routing requires a separately reviewed trace normalizer for your deployment version.

Transport failures are sanitized before rendering because malformed credential headers or upstream protocol errors can include private values. The error message intentionally omits the original exception and response body.


In [ ]:
RUN_LIVE = False
if RUN_LIVE:
    import os
    from urllib.parse import urlsplit

    import httpx

    endpoint = os.environ["WXO_CHAT_URL"]
    try:
        parsed = urlsplit(endpoint)
    except ValueError:
        raise ValueError("HTTPS endpoint without embedded credentials required") from None
    if (
        parsed.scheme != "https"
        or not parsed.hostname
        or parsed.username
        or parsed.password
        or parsed.query
        or parsed.fragment
    ):
        raise ValueError("HTTPS endpoint without embedded credentials required")
    request_headers = {"Authorization": "Bearer " + os.environ["WXO_BEARER_TOKEN"]}
    try:
        with httpx.Client(timeout=90, follow_redirects=False, trust_env=False) as client:
            response = client.post(
                endpoint,
                headers=request_headers,
                json={
                    "messages": [{"role": "user", "content": CASES[0].question}],
                    "stream": False,
                },
            )
    except Exception:
        # Header/protocol errors can echo an invalid credential; never render them.
        raise RuntimeError("Service transport failed; no response was verified") from None
    if not 200 <= response.status_code < 300:
        raise RuntimeError(f"Service request failed with HTTP {response.status_code}")
    try:
        body = response.json()
        answer = body["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError, ValueError):
        raise ValueError("Unexpected Orchestrate chat response") from None
    if not isinstance(answer, str):
        raise ValueError("Expected a text completion")
    print(
        {
            "live_literal_fact_present": CASES[0].required_fact in answer,
            "live_answer_semantics_verified": False,
            "live_routing_verified": False,
        }
    )
else:
    print("No Orchestrate request made. The local trace suite contains synthetic fixture events.")

## Expected local results

The initial three-case suite fails because the safety case routes to the wrong agent even though its final fact is correct. A corrected fixture passes the strict complete-answer contract; an extra tool attempt is still detected. Three contradictory final answers are rejected even though they contain the expected literal fact. Seven malformed trajectory fixtures are rejected, including misplaced tools, premature final answers, unknown failure events, and failed tool calls. No actual deployed agent is claimed repaired.

**Try next:** normalize an authorized synthetic trace from your installed ADK version and preserve every failed tool attempt.


## Interpretation, limitations, and cleanup

Three cases demonstrate evaluator construction, not a statistically meaningful release benchmark. Exact accepted-answer matching is suitable for this authored fixture and can reject valid paraphrases; unrestricted answers require separately defined semantic criteria, independent adjudication and human review. A literal substring match in the live branch establishes neither factual correctness nor a safe release. For an actual ADK evaluation, align the exported case schema and trace fields with the installed ADK version, add repeated multi-turn journeys and tool failures, and establish independent labels. A successful chat HTTP response must never be relabeled as successful internal routing. The notebook does not copy or import a private AgentOps implementation.

The default run keeps its artifacts in memory. Restart the kernel to release them. If you opt into a remote integration, review and remove its synthetic test records using that service's documented controls; this notebook does not delete remote data. Never save credential values or private provider output into notebook cells.


## References and provenance

All scenario text, values, evaluation cases and tutorial code were authored for this public example. No external dataset is bundled. Product names identify optional integrations and do not imply endorsement. API references were checked on 2026-10-01.

- [IBM AgentOps evaluation learning path](https://developer.ibm.com/learningpaths/watsonx-orchestrate-agentops/agent-evaluation-workflows/)
- [Official Orchestrate OpenAPI contract](https://developer.watson-orchestrate.ibm.com/apis/server_openapi.json)
- [IBM Orchestrate ADK](https://developer.watson-orchestrate.ibm.com/)
